# 🏗️ Notebook 1: LinkedIn Connections — Requirements & Architecture

LinkedIn is, at its core, a **giant social graph**. Nodes are people, edges are 'I know this person'.
Almost every interesting feature — *People You May Know*, *degrees of separation*, *who can see your profile* — is a graph algorithm running at planet scale.

In this notebook we:
1. Nail down the **requirements** (what the system must do).
2. Do a quick **back-of-the-envelope** capacity estimate.
3. Sketch a **high-level architecture** and explain the 'why' behind each box.
4. Walk through a **bad → better → best** progression for edge storage.

> Teaching style: small steps, plain language, runnable code. If a cell feels obvious, great — read it and move on.

## 🛠️ Setup

```bash
cd 06-system-designs/linkedin-connections
uv sync
```

Then select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab has **no external services** (no Postgres, no Redis). Everything runs in-process with Python stdlib + `pydantic`, so you can focus on the concepts.

## 1. What are we actually designing?

Think of LinkedIn as a phone book that *also* knows **who knows whom**. The product asks the system four main questions:

| Feature | Question the graph answers |
|---|---|
| Send / accept a request | *Is there an edge between u and v? What is its status?* |
| My connections list | *Who are all neighbors of u?* |
| Degrees of separation | *What is the shortest path between u and v?* |
| People You May Know (PYMK) | *Which non-neighbors share many neighbors with u?* |

Everything else — messaging, feed, search — is built **on top of** this graph.

## 2. Requirements

### Functional
- Send a connection **request** from user `u` to user `v`.
- `v` can **accept** or **reject**. Accepting creates an **undirected edge** `{u, v}`.
- A user can **list** their 1st-degree connections (with pagination).
- A user can **look up** the degree of separation to another user (return `1`, `2`, `3`, or `>3`).
- A user can get a **People You May Know** feed (top-N suggestions).

### Non-functional
- **Reads ≫ writes.** People look at their network constantly; connect rarely. Target ~100:1 read/write ratio.
- **Latency**: 1st-degree list < 50ms p99. PYMK can be **precomputed** offline (seconds → minutes stale is fine).
- **Availability** > strong consistency. 'My new connection doesn't show up for 2 seconds' is OK; a failed page is not.
- **Durability**: never lose an accepted connection.

## 3. Back-of-the-envelope

Let's quickly compute how big the problem is. These numbers matter because they decide whether you need 1 database or 1,000.

In [1]:
# Rough LinkedIn-scale numbers. Adjust and re-run to see how the system grows.
users = 1_000_000_000           # 1B registered users
avg_connections = 500           # median is lower, but the tail is long
bytes_per_edge = 32             # two 8-byte ids + status + timestamp, roughly

# Each connection is ONE relationship, but for O(1) 'list my friends' we usually
# store it twice (once per endpoint). This is the 'symmetric adjacency' trick.
edges_logical  = users * avg_connections // 2
edges_physical = users * avg_connections

storage_gb = edges_physical * bytes_per_edge / 1e9

# Read traffic: say 50 DAU actions that touch the graph per active user per day.
dau = 300_000_000
reads_per_sec = dau * 50 / 86_400
writes_per_sec = reads_per_sec / 100   # ~100:1 read/write ratio

print(f'Logical edges  : {edges_logical:>15,}')
print(f'Physical edges : {edges_physical:>15,}  (symmetric storage)')
print(f'Edge storage   : {storage_gb:>15,.0f} GB  (~{storage_gb/1000:.1f} TB)')
print(f'Reads  / sec   : {reads_per_sec:>15,.0f}')
print(f'Writes / sec   : {writes_per_sec:>15,.0f}')

Logical edges  : 250,000,000,000
Physical edges : 500,000,000,000  (symmetric storage)
Edge storage   :          16,000 GB  (~16.0 TB)
Reads  / sec   :         173,611
Writes / sec   :           1,736


### What the numbers tell us

- **TBs of edges** → one SQL box won't do it. We need **sharding** (split data across many machines).
- **Millions of reads/sec** → we must **cache** hot adjacency lists (Redis-style).
- **Power-law degree distribution** → some users have 30k+ connections ("celebrities"). Their edge list is a **hot key** — caching + fan-out tricks matter a lot.

## 4. High-level architecture

```
          clients (web / mobile)
                 │
                 ▼
          ┌──────────────┐
          │  API Gateway │  auth, rate-limit, routing
          └──────┬───────┘
                 ▼
          ┌──────────────┐     ┌──────────────┐
          │ Graph Service│ ◄──►│ Cache (Redis)│  ← hot adjacency lists
          └──┬───────┬───┘     └──────────────┘
             │       │
     ┌───────▼──┐  ┌─▼──────────┐
     │ Edge DB  │  │ PYMK Store │  ← precomputed top-N per user
     │ (sharded)│  │   (KV)     │
     └──────────┘  └────────────┘
             ▲
             │  nightly batch job (Spark / Flink)
     ┌───────┴───────────┐
     │  Offline PYMK job │  computes 'friends-of-friends' overlap
     └───────────────────┘
```

### Why each box?

- **Graph Service** is stateless → scales horizontally behind a load balancer.
- **Edge DB** is the source of truth. Sharded by `user_id` so a user's whole adjacency list lives on one shard.
- **Cache** absorbs the read storm. TTL is fine; stale by a few seconds is acceptable.
- **PYMK Store** serves suggestions in O(1). Computing them live on every request would hammer the DB.
- **Offline job** does the heavy 'friends-of-friends' math once per day, not per request.

## 5. Edge storage: bad → better → best

The single biggest data-modeling decision is **how to store edges**. Let's walk through three options.

### ❌ Bad — one JSON blob per user

Store a user row with a `connections` column that is a JSON array of friend ids.

- **Write** = read the whole array, append, write it back → **O(N)** per connect/disconnect.
- **Race conditions**: two simultaneous accepts lose one. Requires table locks.
- **Hot user** (30k friends) = a 30k-item JSON blob on every profile view.
- Impossible to index: can't ask *'who are all users connected to user 7?'* without a full scan.

In [2]:
# Demonstration of the blob approach and its pitfalls.
users_blob = {
    1: {'name': 'Ada',  'connections': [2, 3]},
    2: {'name': 'Bob',  'connections': [1]},
    3: {'name': 'Cleo', 'connections': [1]},
}

def connect_blob(users, a, b):
    # Read-modify-write pattern — racy if two callers run at once.
    users[a]['connections'].append(b)
    users[b]['connections'].append(a)

connect_blob(users_blob, 2, 3)
print(users_blob)

# Try this: imagine connect_blob(users, 1, 99) running twice in parallel.
# Both reads see [2, 3], both append 99, both write — we now have [2, 3, 99, 99].
# Duplicate edges are subtle bugs that corrupt counts and recommendations.

{1: {'name': 'Ada', 'connections': [2, 3]}, 2: {'name': 'Bob', 'connections': [1, 3]}, 3: {'name': 'Cleo', 'connections': [1, 2]}}


### 🟡 Better — an edge table, one row per edge

Use a relational table `connections(a, b, since)` with a convention: **always store the smaller id first** (`a < b`). That way each edge exists in exactly one row — no duplicates, easy to count.

- ✅ No duplicates, easy to insert, easy to count.
- ❌ To list 'friends of user 7' you query `WHERE a = 7 OR b = 7` → can't use a single index efficiently.

In [3]:
import sqlite3
con = sqlite3.connect(':memory:')
con.executescript('''
CREATE TABLE connections (
  a INTEGER NOT NULL,
  b INTEGER NOT NULL,
  since TEXT NOT NULL,
  PRIMARY KEY (a, b),
  CHECK (a < b)
);
''')

def add_edge(con, u, v):
    a, b = (u, v) if u < v else (v, u)
    con.execute(
      'INSERT OR IGNORE INTO connections(a,b,since) VALUES(?,?,datetime())',
      (a, b),
    )

for u, v in [(1,2),(2,3),(1,3),(3,4),(1,5)]:
    add_edge(con, u, v)

# 'friends of 3' needs to look at both columns — awkward:
rows = con.execute('SELECT a, b FROM connections WHERE a = 3 OR b = 3').fetchall()
friends_of_3 = [b if a == 3 else a for a, b in rows]
print('friends of 3:', friends_of_3)

friends of 3: [1, 2, 4]


### ✅ Best — symmetric adjacency (two rows per edge)

Store each undirected edge **twice**: `(u, v)` *and* `(v, u)`. Then a single index on `(owner, friend)` answers 'list my friends' in one range scan.

- ✅ O(1) 'friends of X' with a primary-key lookup on `(owner, friend)`.
- ✅ Trivially **shardable** by `owner` — all of user 7's neighbors live on one shard.
- ✅ Pagination is a cursor on `friend` or `created_at`.
- ⚠️ Costs 2× the storage. Worth it: TBs are cheap, latency is not.

This is the model used by Facebook's **TAO**, Twitter's **FlockDB**, and most real-world social graphs.

In [4]:
con2 = sqlite3.connect(':memory:')
con2.executescript('''
CREATE TABLE edges (
  owner  INTEGER NOT NULL,
  friend INTEGER NOT NULL,
  since  TEXT    NOT NULL,
  PRIMARY KEY (owner, friend)
);
CREATE INDEX idx_friend ON edges(friend);
''')

def connect(con, u, v):
    con.execute('INSERT OR IGNORE INTO edges VALUES(?,?,datetime())', (u, v))
    con.execute('INSERT OR IGNORE INTO edges VALUES(?,?,datetime())', (v, u))

for u, v in [(1,2),(2,3),(1,3),(3,4),(1,5)]:
    connect(con2, u, v)

# O(1)-style lookup: single index scan, single shard.
print('friends of 3:', [r[0] for r in con2.execute(
    'SELECT friend FROM edges WHERE owner = 3')])

# Paginated: 'next 10 friends of user 1 after friend_id 2'
print('page(u=1, after=2):', [r[0] for r in con2.execute(
    'SELECT friend FROM edges WHERE owner = 1 AND friend > 2 ORDER BY friend LIMIT 10')])

friends of 3: [1, 2, 4]
page(u=1, after=2): [3, 5]


## 6. Takeaways

- A social network is **just a graph** — and the graph is the hardest part of the design.
- Numbers decide architecture. 500B edges ⇒ sharding + caching + offline PYMK are mandatory, not optional.
- **Store each undirected edge twice** so 'list my friends' is a single index scan.
- Cache hot adjacency lists (Redis) and precompute expensive features (PYMK) offline.

Next: [`02_data_and_api.ipynb`](./02_data_and_api.ipynb) — pydantic models + HTTP API design.